# Semantic Text-to-SQL: Full XiYanSQL-7B QLoRA on Google Colab

Goal: improve semantic correctness, not merely SQL execution success. Train only on 6,601 filtered BIRD TRAIN records with full true BIRD train schema (tables, columns, primary and foreign keys). Freeze a database-disjoint train/validation split. Measure paired base-versus-tuned execution-match on BIRD DEV questions NOT in the previously explored frozen100.

This is a full supervised fine-tuning run, not 64-step diagnostic training. QLoRA uses 4-bit NF4, all-linear LoRA rank 16, explicit completion-only SQL loss and real validation loss.

User must select Runtime > Change runtime type > GPU. A T4 may work at context 1536; L4/A100 is preferred for context 3072. Colab allocates GPU based on availability and account quotas.

Colab GPU training has NOT run merely because this notebook exists. A successful training run requires the verified adapter files AND a positive independently measured paired score. BIRD train license terms and XiYanSQL model attribution apply. Do not share pretrained data or model weights publicly without checking licenses.

Baseline in this repo: XiYan GGUF lexical 58/100, M-Schema 59/100, triple candidate 60/100. Those used a different prompt/runtime than this notebook's HF weights, so the scientifically valid new comparison is HF base vs the SAME HF model plus QLoRA adapter.

In [ ]:
import os, sys, subprocess
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU first"
props = torch.cuda.get_device_properties(0)
VRAM = props.total_memory / 1024**3
assert VRAM >= 14.5, f"7B NF4 training needs an actual NVIDIA GPU; found {VRAM:.1f} GiB"
MAX_LEN = 1536 if VRAM < 20 else 3072
MODEL = "XGenerationLab/XiYanSQL-QwenCoder-7B-2504"
print({"gpu": props.name, "vram_gib":round(VRAM,1), "max_length":MAX_LEN, "model":MODEL})


In [ ]:
# Reproducible source checkout. Do not use current moving main.
import subprocess,sys
from pathlib import Path
REPO = Path("/content/semantic-text2sql-agent")
SOURCE_SHA = "1e1b5ef4c146348229e102c69318d20ff0a30d96"
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/kolihhan/semantic-text2sql-agent.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","--depth","100","origin","research/semantic-sft-colab-20261112"],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SOURCE_SHA],check=True)
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SOURCE_SHA
subprocess.run([sys.executable,"-m","pip","install","-q","-e",f"{REPO}[dev]","transformers>=5.15,<6",
    "peft>=0.20,<1","accelerate>=1.10,<2","bitsandbytes>=0.50.1,<1",
    "datasets>=4","jinja2>=3.1","huggingface_hub>=1.0"],check=True)
print("Pinned source:",SOURCE_SHA)


In [ ]:
# Only BIRD filtered TRAIN gold is used for SFT, never BIRD DEV.
import json,hashlib
from collections import Counter
from pathlib import Path
from huggingface_hub import hf_hub_download
from experiments.semantic_sft_data import prepare_semantic_sft_rows

FILTERED_REPO = "birdsql/bird23-train-filtered"
FILTERED_REV = "4068469807b255fcfc0816bdd520946fe460d256"
SCHEMA_REPO = "Deema/BIRD-SQL"
SCHEMA_REV = "c4a98a1"

train_file = Path(hf_hub_download(FILTERED_REPO,filename="data/train-00000-of-00001.jsonl",repo_type="dataset",revision=FILTERED_REV))
train_tables_file = Path(hf_hub_download(SCHEMA_REPO,filename="train_tables.json",repo_type="dataset",revision=SCHEMA_REV))
dev_tables_file = Path(hf_hub_download(SCHEMA_REPO,filename="dev_tables.json",repo_type="dataset",revision=SCHEMA_REV))
train_raw = [json.loads(line) for line in train_file.read_text().splitlines() if line.strip()]
train_tables = json.loads(train_tables_file.read_text())
dev_tables = json.loads(dev_tables_file.read_text())
assert len(train_raw) == 6601, f"Filtered train revision changed: {len(train_raw)}"
train_rows,val_rows,data_report = prepare_semantic_sft_rows(
    train_raw,train_tables,dev_db_ids={x["db_id"] for x in dev_tables},
    validation_db_fraction=0.15,min_train=1000,min_val=100,
)
assert not set(data_report["train_dbs"]).intersection(data_report["validation_dbs"])
assert not set(data_report["train_dbs"]).intersection({x["db_id"] for x in dev_tables})
data_report["input_sha256"]={k:hashlib.sha256(v.read_bytes()).hexdigest()
    for k,v in (("filtered_train",train_file),("train_tables",train_tables_file),("dev_tables",dev_tables_file))}
print("TRAIN-ONLY SEMANTIC DATA",json.dumps({k:v for k,v in data_report.items() if k not in ("train_dbs","validation_dbs","excluded_dev_dbs")},indent=2))


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
from experiments.semantic_sft_data import tokenize_semantic_pair
tokenizer = AutoTokenizer.from_pretrained(MODEL,trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side="right"

def encode_split(rows):
    prepared,excluded=[],[]
    for row in rows:
        try:
            prepared.append(tokenize_semantic_pair(tokenizer,row,max_length=MAX_LEN))
        except ValueError as exc:
            if "too_long" not in str(exc):
                raise
            excluded.append(row["question_hash"])
    return prepared,excluded

train_tokens,train_excluded=encode_split(train_rows)
val_tokens,val_excluded=encode_split(val_rows)
assert len(train_tokens)>=1000 and len(val_tokens)>=100
assert len(train_excluded)/len(train_rows)<0.35, "Over 35% of training examples exceed GPU context; use L4/A100, not silent truncation"
assert all(any(label != -100 for label in x["labels"]) for x in train_tokens+val_tokens)
print({"train_examples":len(train_tokens),"val_examples":len(val_tokens),
       "train_too_long":len(train_excluded),"val_too_long":len(val_excluded),
       "completion_only_mask":True,"max_length":MAX_LEN})
train_ds=Dataset.from_list(train_tokens)
val_ds=Dataset.from_list(val_tokens)


In [ ]:
# User authorization is needed; Google Drive stores checkpoints and final adapter.
from google.colab import drive
import json
from pathlib import Path
drive.mount("/content/drive")
RUN_NAME = "xiyan7b-semantic-birdfiltered-qlora-v1"
ROOT = Path("/content/drive/MyDrive/semantic-text2sql") / RUN_NAME
ROOT.mkdir(parents=True,exist_ok=True)
contract={
    "model":MODEL,"source_sha":SOURCE_SHA,"filtered_revision":FILTERED_REV,
    "schema_revision":SCHEMA_REV,"max_length":MAX_LEN,
    "train_examples":len(train_tokens),"val_examples":len(val_tokens),
    "learning_rate":5e-5,"rank":16,"lora_alpha":32,"lora_dropout":0.05,
    "epochs":1.0,"gradient_accumulation":8,"train_db_split":"0.15",
    "input_sha256":data_report["input_sha256"],
}
contract_file=ROOT/"run-contract.json"
if contract_file.exists():
    assert json.loads(contract_file.read_text())==contract, "Existing run differs. Change RUN_NAME; never silently overwrite."
else:
    contract_file.write_text(json.dumps(contract,indent=2,sort_keys=True))
( ROOT / "data-audit.json" ).write_text(json.dumps({**data_report,"excluded_train_length":len(train_excluded),
    "excluded_val_length":len(val_excluded)},indent=2,sort_keys=True))
print("Persistent output:",ROOT)


In [ ]:
# Full all-linear 7B QLoRA; no 64-step shortcut.
import torch,os,random
from transformers import (AutoModelForCausalLM,BitsAndBytesConfig,TrainingArguments,
                          Trainer,EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.trainer_utils import get_last_checkpoint

if (ROOT/"adapter_model.safetensors").exists():
    raise RuntimeError("Final adapter already exists. Do not overwrite; open a new RUN_NAME.")
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
base=AutoModelForCausalLM.from_pretrained(MODEL,quantization_config=quant,
    device_map="auto",dtype=dtype,trust_remote_code=True)
base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True)
peft_config=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,
    target_modules="all-linear",task_type="CAUSAL_LM",bias="none")
model=get_peft_model(base,peft_config)
assert sum(p.numel() for p in model.parameters() if p.requires_grad)>0
assert all("lora_" in n for n,p in model.named_parameters() if p.requires_grad)
model.config.use_cache=False

def collate(batch):
    max_len = ((max(len(x["input_ids"]) for x in batch)+7)//8)*8
    ids=torch.full((len(batch),max_len),tokenizer.pad_token_id,dtype=torch.long)
    att=torch.zeros((len(batch),max_len),dtype=torch.long)
    labels=torch.full((len(batch),max_len),-100,dtype=torch.long)
    for i,row in enumerate(batch):
        length=len(row["input_ids"])
        ids[i,:length]=torch.tensor(row["input_ids"],dtype=torch.long)
        att[i,:length]=1
        labels[i,:length]=torch.tensor(row["labels"],dtype=torch.long)
    return {"input_ids":ids,"attention_mask":att,"labels":labels}

args=TrainingArguments(
    output_dir=str(ROOT/"checkpoints"),
    num_train_epochs=1.0,per_device_train_batch_size=1,per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,learning_rate=5e-5,lr_scheduler_type="cosine",
    warmup_ratio=0.05,weight_decay=0,logging_steps=25,
    eval_strategy="steps",eval_steps=100,
    save_strategy="steps",save_steps=100,save_total_limit=2,
    load_best_model_at_end=True,metric_for_best_model="eval_loss",
    greater_is_better=False,gradient_checkpointing=True,
    bf16=(dtype==torch.bfloat16),fp16=(dtype==torch.float16),
    optim="paged_adamw_8bit",report_to="none",seed=42,data_seed=42,
    remove_unused_columns=False,
)
trainer=Trainer(model=model,args=args,train_dataset=train_ds,
    eval_dataset=val_ds.select(range(min(128,len(val_ds)))),data_collator=collate,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
ckpt=get_last_checkpoint(str(ROOT/"checkpoints")) if (ROOT/"checkpoints").exists() else None
result=trainer.train(resume_from_checkpoint=ckpt)
assert trainer.state.global_step>64, "Full training did not advance beyond diagnostic budget"
trainer.save_model(str(ROOT))
tokenizer.save_pretrained(str(ROOT))
trainer.save_state()
(ROOT/"training-metrics.json").write_text(json.dumps({"metrics":result.metrics,
    "global_step":trainer.state.global_step,"contract":contract},indent=2,default=str))
assert (ROOT/"adapter_model.safetensors").is_file()
print("REAL_FULL_QLORA_FINISHED",{"steps":trainer.state.global_step,"adapter":str(ROOT/"adapter_model.safetensors")})


In [ ]:
# BIRD DEV is downloaded ONLY AFTER training finishes. Check checksum.
import urllib.request,hashlib,zipfile,os,shutil
from pathlib import Path
DEVZIP=Path("/content/bird-dev-official.zip")
if not DEVZIP.exists():
    urllib.request.urlretrieve(
        "https://huggingface.co/datasets/HAL-9001/bird-dev/resolve/main/dev.zip",
        DEVZIP)
expected="cdd6d19faeb45a23970b98d3ef6c40a87987c95459c2cf12076897a60cf5a630"
assert hashlib.sha256(DEVZIP.read_bytes()).hexdigest()==expected,"BIRD DEV archive digest mismatch"
DEVROOT=Path("/content/bird-dev-unpacked")
if not DEVROOT.exists():
    with zipfile.ZipFile(DEVZIP) as z:
        z.extractall(DEVROOT)
dev_json_candidates=list(DEVROOT.rglob("dev.json"))
dev_tables_candidates=list(DEVROOT.rglob("dev_tables.json"))
db_zip_candidates=list(DEVROOT.rglob("dev_databases.zip"))
assert len(dev_json_candidates)==len(dev_tables_candidates)==len(db_zip_candidates)==1
with zipfile.ZipFile(db_zip_candidates[0]) as z:
    z.extractall(DEVROOT/"databases")
sql_db_candidates=list(DEVROOT.rglob("dev_databases"))
sql_db_candidates=[p for p in sql_db_candidates if any(p.rglob("*.sqlite"))]
assert sql_db_candidates,"BIRD dev databases not extracted"
DB_ROOT=sql_db_candidates[0]
dev_rows=json.loads(dev_json_candidates[0].read_text())
dev_schema=json.loads(dev_tables_candidates[0].read_text())
frozen=json.loads((REPO/"runs/bird-paired-dev-v4/paired.json").read_text())
FROZEN={str(x["case_id"]) for x in frozen["cases"]}
assert len(FROZEN)==100 and len(dev_rows)>=1500
print("Evaluation:",len(dev_rows),"BIRD DEV questions, frozen100 excluded:",len(FROZEN))


In [ ]:
# True correctness test. Base and tuned use identical HF weights/runtime/prompt.
# New independent 100 questions; read-only SQL result match, not SQL execution success.
import gc,torch,json
from experiments.semantic_sft_eval import run_inference,compare_paired_scores
BASE_OUTPUT=ROOT/"independent100-base.jsonl"
TUNED_OUTPUT=ROOT/"independent100-tuned.jsonl"
PAIRED_OUTPUT=ROOT/"independent100-paired.json"
# Release training model before loading inference weights. Runtime may need restart if CUDA fragmented.
del trainer,model,base
gc.collect()
torch.cuda.empty_cache()
if not BASE_OUTPUT.exists():
    base_summary=run_inference(model_id=MODEL,adapter_path=None,data=dev_rows,
      schemas=dev_schema,db_root=DB_ROOT,excluded_ids=FROZEN,count=100,output=BASE_OUTPUT)
gc.collect()
torch.cuda.empty_cache()
if not TUNED_OUTPUT.exists():
    tuned_summary=run_inference(model_id=MODEL,adapter_path=ROOT,data=dev_rows,
      schemas=dev_schema,db_root=DB_ROOT,excluded_ids=FROZEN,count=100,output=TUNED_OUTPUT)
base_results=[json.loads(line) for line in BASE_OUTPUT.read_text().splitlines() if line.strip()]
tuned_results=[json.loads(line) for line in TUNED_OUTPUT.read_text().splitlines() if line.strip()]
paired=compare_paired_scores(base_results,tuned_results,min_gain=3)
PAIRED_OUTPUT.write_text(json.dumps(paired,indent=2,sort_keys=True))
print("PAIRED_SEMANTIC_EXECUTION_MATCH",json.dumps(paired,indent=2,sort_keys=True))
if not paired["gate_met"]:
    print("No promotion. Model needs further research on new TRAIN data, not tuning on these DEV answers.")


## Decision rule

Use the same 100 independently predeclared BIRD DEV questions for base and tuned model; historical frozen100 excluded. Only a net execution-match gain of at least +3 correct questions passes the preregistered local gate. A positive training loss curve alone is not sufficient.

Training input uses only BIRD train filtered data, schema, question and annotated evidence. It NEVER puts gold SQL or dev SQL into the user prompt, although gold SQL is used as the supervised training target. The result is a research experiment, not an official BIRD leaderboard submission.

If a GPU is not allocated, the run cannot execute; no training or score should be claimed.